# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides an example workflow for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema, available at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# List the available record sets, their @id, and fields.
from typing import List

def print_record_sets(ds):
    print("Available Record Sets:")
    for rset in ds.metadata.record_sets:
        print(f"- {rset['@id']}")
        fields = rset.get('fields', [])
        if fields:
            print("  Fields:")
            for f in fields:
                print(f"    - {f['@id']} ({f['name'] if 'name' in f else ''})")
        print()

# Print all record sets and their fields with @id
print_record_sets(dataset)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

**Note:** Use `@id` values from the overview above to reference record sets and fields.

In [ ]:
# Collect all record set @id values
record_sets = [recset["@id"] for recset in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded '{record_set_id}' with shape {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for record set '{record_set_id}'")

# Example: Show columns and first rows for the first available DataFrame
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No data available in any record set.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalization, and grouping.

For demonstration, we'll select a numeric field from one record set, filter and process data using its `@id`.

In [ ]:
# You'll need to adapt these @id values to real ones present in your dataset.
# For demonstration, we'll proceed if at least one DataFrame is available.

if dataframes:
    # Pick the first available record set as an example
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Attempt to detect a numeric column for demonstration
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        print(f"Using numeric field: {numeric_field_id}\n")
        threshold = df[numeric_field_id].quantile(0.75) if not df[numeric_field_id].isnull().all() else 0
        # Filter records where the numeric field > threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        # Normalize the numeric field
        filtered_df = filtered_df.copy()  # Avoid SettingWithCopyWarning
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by some non-numeric field if available
        non_num_cols = [col for col in df.columns if df[col].dtype == object]
        group_field = non_num_cols[0] if non_num_cols else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric columns found for processing.")
else:
    print("No data available for analysis.")

## 5. Visualization
Visualize the distribution of a numeric field from the selected record set.

In [ ]:
# Visualization example: Histogram and Boxplot of the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, color='royalblue')
    plt.title(f'Distribution of {numeric_field_id}')

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field_id], color='lightgreen')
    plt.title(f'Boxplot of {numeric_field_id}')
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and begin exploring a Croissant-structured dataset using `mlcroissant`. We performed basic data loading, filtering, normalization, grouping, and visualization—all by referencing record sets and fields by their `@id`.

For more advanced analysis, you can:
- Explore additional record sets and their relationships by `@id`,
- Join data from different record sets (if linked),
- Apply statistical or machine learning methods to the cleaned data.

Refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) for more details and examples.